In [0]:
storage_account_name = "jana60305219"
storage_account_key = "0GM7spdOovemizH7Y7FRNCvl/bLz5y0odTdC9mhp2Lm4/cxI9htWaDjHJU6iBEZlvhmmY8ymw3ic+AStT8nfqQ=="
spark.conf.set(
 f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net",
 storage_account_key
)

In [0]:
reviews_path = "abfss://processed@jana60305219.dfs.core.windows.net/reviews/"
reviews_df = spark.read.parquet(reviews_path)


In [0]:
spark.conf.set("fs.azure.account.key.jana60305219.dfs.core.windows.net", "0GM7spdOovemizH7Y7FRNCvl/bLz5y0odTdC9mhp2Lm4/cxI9htWaDjHJU6iBEZlvhmmY8ymw3ic+AStT8nfqQ==")
reviews_df.printSchema()

display(reviews_df.limit(5))

root
 |-- reviewerID: string (nullable = true)
 |-- asin: string (nullable = true)
 |-- reviewerName: string (nullable = true)
 |-- helpful: array (nullable = true)
 |    |-- element: integer (containsNull = true)
 |-- reviewText: string (nullable = true)
 |-- overall: double (nullable = true)
 |-- summary: string (nullable = true)
 |-- unixReviewTime: integer (nullable = true)
 |-- reviewTime: string (nullable = true)
 |-- review_year: integer (nullable = true)



reviewerID,asin,reviewerName,helpful,reviewText,overall,summary,unixReviewTime,reviewTime,review_year
A16KHL9MX9FXV3,B00BDCF5JK,Scott Miller,"List(4, 4)","I purchased the ArmorSuit MilitaryShield for my brand new Microsoft Surface Pro 2 (yes, although this product fits Surface Pro (1), it also perfectly fits the Surface Pro 2). It took me about 30 minutes to apply the full body protection to the Surface. If you have ever installed an Invisible Shield or similar ""wet application"" screen protection you will be right at home with the ArmorSuit. They supplied a generous amount of liquid for application of the screen protector, I only used half of the bottle and I was very liberal with my application during installation.My installation tips would be to lay down a microfiber cloth to rest your surface on, and prepare your work area near a sunny window or a bright light so you can check the screen for dust (the reflection from the lights helps you see the particles). I also had a spare microfiber cloth handy during installation so I could quickly blot out the liquid when you squeegee the screen protectors in place.I did make a few mistakes while applying the protectors, however I was able to easily lift the pieces off or slide them with relative ease.There were some micro-bubbles after installation, and as of day 2 they have *mostly* disappeared. I do have a few streaks from the squeegee on the front screen protector that I am hoping will go away in a few days.I can confirm that the screen protector is slightly sticky when you use either your finger or digitizer pen. It may be my imagination but I feel like it became a little bit less sticky overnight. It does not have the ""orange peel"" look of the Invisible Shields.My primary use for the Surface will be taking notes in Microsoft OneNote using the stylus / digitizer. I can confirm that what others have said about the stylus leaving marks on the screen protector, however you cannot see these marks when the device is on. I also find the texture to be more pleasant to write on with the stylus than the bare glass, but each person will have their own preference. I found it gave me just enough resistance to feel like paper, and to give me better control over my pen strokes. I found the bare glass to be too slippery.The back, kickstand, and side pieces are very well cut out and the tolerances are very good. There is maybe a half millimeter of clearance on the outside edges of all the pieces, which in my book is full coverage.**Update**While cleaning my Surface Pro 2, I decided to try to improve the feel of the screen protector. I sprayed some Meguiar's Supreme Shine Hi-Gloss Protectant on the front screen protector and rubbed it in with a microfiber cloth. This drastically improved the feel of the screen protector, and it is now much more glass like both to your fingers and with the digitizer. I'll update this review if anything horrible happens, but so far it is working great.",5.0,Very good protection for Surface Pro 2,1383955200,"11 9, 2013",2013
ANUF4LWX96DXW,B00BDCF716,Bau Kieu,"List(1, 2)","Only the carbon fiber body wrap is good (excellent, in fact). The screen protector is not good. It is too soft for the stylus and fingers as well. I had to remove it after one day trying.",3.0,Half Good,1377129600,"08 22, 2013",2013
A20RSO8H7Z772Y,B00BDCF716,droid owner,"List(4, 4)",The CF looks great and provides some grip. The screen protector is good but the keyboard on the surface does put some lines on it. I can't really see the lines while using the Pro but I can see them when the unit is off. It also seems like the type keyboard cover doesn't sit as snuggly as it used to.CF = 5 starsSP = 3 stars,4.0,"CF is awesome, Screen protector is ok",1370476800,"06 6, 2013",2013
A1THQXOSQUPPNP,B00BDCF716,Kindle Customer,"List(1, 1)","My pen scratched it up when using the screen digitizer, so I pulled it off. But the carbon fiber back is amazing, thick and tough. I love it and it protects my Surface 

In [0]:
from pyspark.sql.functions import col, trim, length
clean_reviews_df = reviews_df
# Drop rows with missing critical fields
clean_reviews_df = clean_reviews_df.filter(
 col("asin").isNotNull() &
 col("reviewerID").isNotNull() &
 col("overall").isNotNull()
)
# Enforce valid rating values
clean_reviews_df = clean_reviews_df.filter(
 (col("overall") >= 1) & (col("overall") <= 5)
)
# Clean review text
clean_reviews_df = clean_reviews_df.withColumn(
 "reviewText",
 trim(col("reviewText"))
).filter(
 length(col("reviewText")) >= 10
)

In [0]:
print("Rows before cleaning:", reviews_df.count())
print("Rows after cleaning:", clean_reviews_df.count())
display(clean_reviews_df.limit(5))

Rows before cleaning: 1689188
Rows after cleaning: 1687427


reviewerID,asin,reviewerName,helpful,reviewText,overall,summary,unixReviewTime,reviewTime,review_year
A16KHL9MX9FXV3,B00BDCF5JK,Scott Miller,"List(4, 4)","I purchased the ArmorSuit MilitaryShield for my brand new Microsoft Surface Pro 2 (yes, although this product fits Surface Pro (1), it also perfectly fits the Surface Pro 2). It took me about 30 minutes to apply the full body protection to the Surface. If you have ever installed an Invisible Shield or similar ""wet application"" screen protection you will be right at home with the ArmorSuit. They supplied a generous amount of liquid for application of the screen protector, I only used half of the bottle and I was very liberal with my application during installation.My installation tips would be to lay down a microfiber cloth to rest your surface on, and prepare your work area near a sunny window or a bright light so you can check the screen for dust (the reflection from the lights helps you see the particles). I also had a spare microfiber cloth handy during installation so I could quickly blot out the liquid when you squeegee the screen protectors in place.I did make a few mistakes while applying the protectors, however I was able to easily lift the pieces off or slide them with relative ease.There were some micro-bubbles after installation, and as of day 2 they have *mostly* disappeared. I do have a few streaks from the squeegee on the front screen protector that I am hoping will go away in a few days.I can confirm that the screen protector is slightly sticky when you use either your finger or digitizer pen. It may be my imagination but I feel like it became a little bit less sticky overnight. It does not have the ""orange peel"" look of the Invisible Shields.My primary use for the Surface will be taking notes in Microsoft OneNote using the stylus / digitizer. I can confirm that what others have said about the stylus leaving marks on the screen protector, however you cannot see these marks when the device is on. I also find the texture to be more pleasant to write on with the stylus than the bare glass, but each person will have their own preference. I found it gave me just enough resistance to feel like paper, and to give me better control over my pen strokes. I found the bare glass to be too slippery.The back, kickstand, and side pieces are very well cut out and the tolerances are very good. There is maybe a half millimeter of clearance on the outside edges of all the pieces, which in my book is full coverage.**Update**While cleaning my Surface Pro 2, I decided to try to improve the feel of the screen protector. I sprayed some Meguiar's Supreme Shine Hi-Gloss Protectant on the front screen protector and rubbed it in with a microfiber cloth. This drastically improved the feel of the screen protector, and it is now much more glass like both to your fingers and with the digitizer. I'll update this review if anything horrible happens, but so far it is working great.",5.0,Very good protection for Surface Pro 2,1383955200,"11 9, 2013",2013
ANUF4LWX96DXW,B00BDCF716,Bau Kieu,"List(1, 2)","Only the carbon fiber body wrap is good (excellent, in fact). The screen protector is not good. It is too soft for the stylus and fingers as well. I had to remove it after one day trying.",3.0,Half Good,1377129600,"08 22, 2013",2013
A20RSO8H7Z772Y,B00BDCF716,droid owner,"List(4, 4)",The CF looks great and provides some grip. The screen protector is good but the keyboard on the surface does put some lines on it. I can't really see the lines while using the Pro but I can see them when the unit is off. It also seems like the type keyboard cover doesn't sit as snuggly as it used to.CF = 5 starsSP = 3 stars,4.0,"CF is awesome, Screen protector is ok",1370476800,"06 6, 2013",2013
A1THQXOSQUPPNP,B00BDCF716,Kindle Customer,"List(1, 1)","My pen scratched it up when using the screen digitizer, so I pulled it off. But the carbon fiber back is amazing, thick and tough. I love it and it protects my Surface 

In [0]:
clean_reviews_path = "abfss://processed@jana60305219.dfs.core.windows.net/clean_reviews/"
clean_reviews_df.write.mode("overwrite").parquet(clean_reviews_path)